# 1. Linear Regression

**Machine Learning Fundamentals and Predictive Analytics — Notebook 1 of 11**

Linear regression is the "hello world" of machine learning, and it never stops being useful.
It is fast, it is interpretable, it needs no tuning to get a first answer, and it is the
baseline every fancier model has to beat.

Notebook 9 of the statistics module covered the *statistical* view: least squares, coefficient
inference, residual diagnostics. This notebook takes the *machine-learning* view: predicting
well on unseen data, engineering features, and building a pipeline you can put in production.

### What you will learn

1. The model, and the two ways to fit it: **normal equation** and **gradient descent**
2. The standard scikit-learn workflow, end to end
3. Metrics: **MAE, MSE, RMSE, $R^2$, MAPE** — and which to report
4. **Feature scaling** and when it matters
5. **Feature engineering**: polynomials, interactions, log transforms, categorical encoding
6. **Regularisation**: Ridge, Lasso, Elastic Net, and choosing $\alpha$ by CV
7. Interpreting coefficients honestly
8. A complete case study with a `Pipeline` and `ColumnTransformer`
9. When linear regression is the wrong tool

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet,
                                 RidgeCV, LassoCV, ElasticNetCV, SGDRegressor,
                                 HuberRegressor)
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             mean_absolute_percentage_error)
from sklearn.dummy import DummyRegressor

rng = np.random.default_rng(seed=1)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
CV = KFold(5, shuffle=True, random_state=0)

---
## 1.1 The model

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p = \mathbf{x}^\top\boldsymbol\beta$$

We choose $\boldsymbol\beta$ to minimise the **mean squared error** on the training data:

$$J(\boldsymbol\beta) = \frac{1}{n}\sum_{i=1}^{n}\big(y_i - \mathbf{x}_i^\top\boldsymbol\beta\big)^2$$

Two ways to solve it:

| | Normal equation | Gradient descent |
|---|---|---|
| Formula | $\hat{\boldsymbol\beta} = (X^\top X)^{-1}X^\top\mathbf{y}$ | $\boldsymbol\beta \leftarrow \boldsymbol\beta - \eta\nabla J$ |
| Cost | $O(p^3)$ — one shot, exact | $O(np)$ per step — iterative, approximate |
| Good when | $p$ is small (say < 10,000) | $p$ or $n$ is huge, or data streams in |
| Needs scaling? | No | **Yes** |
| Used by | `LinearRegression` | `SGDRegressor`, every neural network |

`LinearRegression` in scikit-learn actually uses an SVD-based least-squares solver rather than
inverting $X^\top X$ directly — numerically safer, same answer.

In [ ]:
# Data: predicting house price from size, age and distance to city centre
n = 400
size = rng.uniform(50, 260, n)                 # square metres
age = rng.uniform(0, 45, n)                    # years
distance = rng.uniform(0.5, 25, n)             # km from centre

price = (25_000 + 2_400*size - 900*age - 1_800*distance
         + rng.normal(0, 30_000, n))           # in currency units

homes = pd.DataFrame({"size": size, "age": age, "distance": distance, "price": price})
print(homes.describe().round(1))
print(f"\nTrue coefficients: size=+2400, age=-900, distance=-1800, intercept=25000")

In [ ]:
FEATURES = ["size", "age", "distance"]
X = homes[FEATURES].to_numpy()
y = homes["price"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(f"train {X_train.shape}, test {X_test.shape}\n")

# --- Method 1: the normal equation, by hand
X1 = np.column_stack([np.ones(len(X_train)), X_train])
beta = np.linalg.solve(X1.T @ X1, X1.T @ y_train)
print("Normal equation :", np.round(beta, 1))

# --- Method 2: scikit-learn
lin = LinearRegression().fit(X_train, y_train)
print("LinearRegression:", np.round(np.r_[lin.intercept_, lin.coef_], 1))

# --- Method 3: gradient descent, by hand (scaled features are essential)
scaler = StandardScaler().fit(X_train)
Xs = scaler.transform(X_train)
Xs1 = np.column_stack([np.ones(len(Xs)), Xs])

def gradient_descent(Xb, yv, lr=0.1, n_iter=400):
    '''Batch gradient descent on the mean squared error.'''
    b = np.zeros(Xb.shape[1])
    history = []
    m = len(yv)
    for _ in range(n_iter):
        pred = Xb @ b
        grad = -2.0 / m * Xb.T @ (yv - pred)
        b -= lr * grad
        history.append(((yv - Xb @ b) ** 2).mean())
    return b, history

b_gd, hist = gradient_descent(Xs1, y_train)
# Convert the scaled coefficients back to the original units
coef_orig = b_gd[1:] / scaler.scale_
inter_orig = b_gd[0] - (b_gd[1:] * scaler.mean_ / scaler.scale_).sum()
print("Gradient descent:", np.round(np.r_[inter_orig, coef_orig], 1))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(hist, color="steelblue", lw=2)
ax[0].set_xlabel("iteration"); ax[0].set_ylabel("training MSE")
ax[0].set_yscale("log"); ax[0].set_title("Gradient descent converges")

# Learning rate matters
for lr, colour in [(0.005, "steelblue"), (0.05, "seagreen"), (0.3, "darkorange"),
                   (0.95, "crimson")]:
    _, h = gradient_descent(Xs1, y_train, lr=lr, n_iter=120)
    ax[1].plot(h, color=colour, lw=1.8, label=f"lr = {lr}")
ax[1].set_yscale("log"); ax[1].set_xlabel("iteration"); ax[1].set_ylabel("training MSE")
ax[1].set_title("Learning rate: too small crawls, too large oscillates")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

# SGDRegressor: gradient descent as a scikit-learn estimator (needs scaling)
sgd = make_pipeline(StandardScaler(), SGDRegressor(max_iter=5000, tol=1e-6, random_state=0))
sgd.fit(X_train, y_train)
print(f"LinearRegression test R^2 : {lin.score(X_test, y_test):.4f}")
print(f"SGDRegressor     test R^2 : {sgd.score(X_test, y_test):.4f}")
print("\nSame answer. SGD exists for the cases where the closed form is too expensive.")

---
## 1.2 Metrics: which number to report

| Metric | Formula | Units | Reads as |
|---|---|---|---|
| **MAE** | $\frac{1}{n}\sum\|y-\hat{y}\|$ | same as $y$ | typical error; robust to outliers |
| **MSE** | $\frac{1}{n}\sum(y-\hat{y})^2$ | $y^2$ | for optimisation, not for reporting |
| **RMSE** | $\sqrt{\text{MSE}}$ | same as $y$ | error, with big misses weighted heavily |
| **$R^2$** | $1-\text{SSE}/\text{SST}$ | unitless | fraction of variance explained |
| **MAPE** | $\frac{100}{n}\sum\|\frac{y-\hat{y}}{y}\|$ | % | relative error; explodes when $y\approx 0$ |

Practical guidance:

- Report **RMSE or MAE** (in units your stakeholder understands) **plus $R^2$**.
- Choose RMSE if large errors are disproportionately costly, MAE if all errors cost the same
  per unit.
- **Always compare against a baseline.** `DummyRegressor` predicting the mean gives
  $R^2 = 0$ by definition; that is the bar.
- A negative test $R^2$ means you are worse than predicting the mean. It happens.

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te, name):
    pred_tr, pred_te = model.predict(X_tr), model.predict(X_te)
    return {
        "model": name,
        "train_RMSE": np.sqrt(mean_squared_error(y_tr, pred_tr)),
        "test_RMSE": np.sqrt(mean_squared_error(y_te, pred_te)),
        "test_MAE": mean_absolute_error(y_te, pred_te),
        "test_R2": r2_score(y_te, pred_te),
        "test_MAPE_%": mean_absolute_percentage_error(y_te, pred_te) * 100,
    }

rows = [
    evaluate(DummyRegressor(strategy="mean").fit(X_train, y_train),
             X_train, y_train, X_test, y_test, "baseline (predict the mean)"),
    evaluate(lin, X_train, y_train, X_test, y_test, "linear regression"),
]
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"\nThe target has sd {y.std():,.0f}, so an RMSE of "
      f"{rows[1]['test_RMSE']:,.0f} means we have removed most of the variation.")
print(f"True noise level in the simulation: 30,000 -- we are close to the floor.")

In [ ]:
# Diagnostics you should look at for every regression model
pred_test = lin.predict(X_test)
resid = y_test - pred_test

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].scatter(y_test, pred_test, s=20, alpha=0.7, color="steelblue")
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
ax[0].plot(lims, lims, "k--", lw=1.4)
ax[0].set_xlabel("actual price"); ax[0].set_ylabel("predicted price")
ax[0].set_title(f"Predicted vs actual (R^2 = {r2_score(y_test, pred_test):.3f})")

ax[1].scatter(pred_test, resid, s=20, alpha=0.7, color="steelblue")
ax[1].axhline(0, color="crimson", lw=1.4)
ax[1].set_xlabel("predicted"); ax[1].set_ylabel("residual")
ax[1].set_title("Residuals: want a flat, structureless band")

ax[2].hist(resid, bins=25, color="steelblue", edgecolor="white")
ax[2].axvline(0, color="crimson", lw=1.4)
ax[2].set_xlabel("residual"); ax[2].set_title("Residual distribution")
plt.tight_layout(); plt.show()

print(f"Mean residual   : {resid.mean():,.1f}  (want ~0: no systematic bias)")
print(f"Residual sd     : {resid.std():,.1f}")
print(f"Worst under-prediction: {resid.max():,.0f}")
print(f"Worst over-prediction : {resid.min():,.0f}")

---
## 1.3 Does linear regression need feature scaling?

**For plain OLS: no.** The closed-form solution is invariant to linear rescaling of the
features — the coefficients simply change units to compensate.

**But yes for all of these:**

- **Gradient descent / `SGDRegressor`** — unscaled features make the loss surface a long
  narrow valley, and convergence crawls
- **Ridge / Lasso / Elastic Net** — the penalty compares coefficients directly, so it treats
  features unequally unless their scales match
- **Comparing coefficient magnitudes** — a coefficient of 2,400 per m² and 900 per year are
  not comparable until you standardise

In [ ]:
# Proof that OLS is scale-invariant, and that Ridge is not
X_scaled_train = StandardScaler().fit(X_train).transform(X_train)
X_scaled_test = StandardScaler().fit(X_train).transform(X_test)

ols_raw = LinearRegression().fit(X_train, y_train)
ols_scaled = LinearRegression().fit(X_scaled_train, y_train)
ridge_raw = Ridge(alpha=100).fit(X_train, y_train)
ridge_scaled = make_pipeline(StandardScaler(), Ridge(alpha=100)).fit(X_train, y_train)

print(f"OLS on raw features   : test R^2 = {ols_raw.score(X_test, y_test):.6f}")
print(f"OLS on scaled features: test R^2 = {ols_scaled.score(X_scaled_test, y_test):.6f}")
print("  -> identical, to many decimal places\n")
print(f"Ridge on raw features   : test R^2 = {ridge_raw.score(X_test, y_test):.6f}")
print(f"Ridge on scaled features: test R^2 = {ridge_scaled.score(X_test, y_test):.6f}")
print("  -> different, because the penalty depends on the units\n")

print("Standardised coefficients ARE comparable:")
for f_, c in zip(FEATURES, ols_scaled.coef_):
    print(f"  {f_:<9} {c:>12,.0f} per standard deviation")
print("\nSo size matters most, then distance, then age -- which raw coefficients")
print("(2400, -900, -1800) would have told you incorrectly.")

---
## 1.4 Feature engineering

Linear regression can only add things up. **Feature engineering** is how you give it
non-linear shapes to add up. This is usually where the biggest gains come from — larger than
any change of algorithm.

| Situation | Engineered feature |
|---|---|
| Diminishing returns | $\log(x)$, $\sqrt{x}$ |
| U-shaped or curved effect | $x^2$, $x^3$ |
| Effect of $x_1$ depends on $x_2$ | $x_1 \cdot x_2$ |
| Multiplicative outcome (prices, counts) | model $\log(y)$ |
| Category | one-hot / target encoding |
| Ratios that mean something | price per m², spend per visit |
| Dates | day-of-week, month, is-holiday, days-since-event |

In [ ]:
# A dataset where the naive linear model fails and engineering fixes it
m = 500
area = rng.uniform(40, 300, m)
rooms = rng.integers(1, 6, m)
luxury = rng.integers(0, 2, m)

# Truth: log-linear in area, plus an interaction between rooms and luxury
true_price = np.exp(10.0 + 0.45*np.log(area) + 0.06*rooms + 0.18*luxury
                    + 0.05*rooms*luxury + rng.normal(0, 0.13, m))

hp = pd.DataFrame({"area": area, "rooms": rooms, "luxury": luxury, "price": true_price})

designs = {
    "raw features": ["area", "rooms", "luxury"],
    "+ log(area)": ["area", "rooms", "luxury", "log_area"],
    "+ interaction": ["area", "rooms", "luxury", "log_area", "rooms_x_luxury"],
}
hp["log_area"] = np.log(hp.area)
hp["rooms_x_luxury"] = hp.rooms * hp.luxury

for name, cols in designs.items():
    sc = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()),
                         hp[cols], hp.price, cv=CV, scoring="r2").mean()
    print(f"  {name:<16} CV R^2 = {sc:.4f}")

# Modelling log(price) instead of price -- the right move for a multiplicative target
sc_log = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()),
                         hp[designs["+ interaction"]], np.log(hp.price),
                         cv=CV, scoring="r2").mean()
print(f"  {'predicting log(price)':<16} CV R^2 = {sc_log:.4f}   <- on the log scale")
print("\nEach engineered feature is a hypothesis about the mechanism. Adding the log and")
print("the interaction did more for this model than any algorithm change could.")

In [ ]:
# PolynomialFeatures generates powers and interactions automatically
base_cols = ["area", "rooms", "luxury"]
print(f"{'degree':>8} {'n features':>12} {'CV R^2':>10}")
for d in (1, 2, 3, 4):
    pipe = make_pipeline(PolynomialFeatures(d, include_bias=False),
                         StandardScaler(), Ridge(alpha=1.0))
    n_feat = PolynomialFeatures(d, include_bias=False).fit(hp[base_cols]).n_output_features_
    sc = cross_val_score(pipe, hp[base_cols], np.log(hp.price), cv=CV, scoring="r2").mean()
    print(f"{d:>8} {n_feat:>12} {sc:>10.4f}")
print("\nUseful, but it grows combinatorially and produces uninterpretable columns.")
print("Prefer a handful of features you can explain, generated from domain knowledge.")

---
## 1.5 Regularisation in practice

From Notebook 11 of the statistics module: **Ridge** ($L_2$) shrinks coefficients smoothly,
**Lasso** ($L_1$) zeroes some of them out, **Elastic Net** does both.

$$\text{Ridge: } \text{MSE} + \alpha\sum\beta_j^2 \qquad
\text{Lasso: } \text{MSE} + \alpha\sum|\beta_j| \qquad
\text{ElasticNet: } \text{MSE} + \alpha\left(\rho\sum|\beta_j| + \tfrac{1-\rho}{2}\sum\beta_j^2\right)$$

When to reach for which:

| Situation | Use |
|---|---|
| Many correlated features, want all of them | **Ridge** |
| Many features, suspect most are useless | **Lasso** |
| Both of the above | **Elastic Net** |
| $p > n$ | any of them — OLS is not even defined |
| Need interpretability | Lasso (it hands you a short list) |

Always inside a `Pipeline` with a scaler, and always choose $\alpha$ by cross-validation.

In [ ]:
# A realistic mess: 80 features, 12 informative, several highly correlated, n = 150
m2, p2, k2 = 150, 80, 12
Xr = rng.normal(size=(m2, p2))
Xr[:, 1] = Xr[:, 0] + rng.normal(0, 0.1, m2)         # near-duplicate features
Xr[:, 2] = Xr[:, 0] + rng.normal(0, 0.1, m2)
beta_true = np.zeros(p2)
beta_true[:k2] = rng.normal(0, 3, k2)
yr = Xr @ beta_true + rng.normal(0, 2.0, m2)

Xa, Xb, ya, yb = train_test_split(Xr, yr, test_size=0.3, random_state=0)

candidates = {
    "OLS":          LinearRegression(),
    "Ridge (CV)":   RidgeCV(alphas=np.logspace(-3, 3, 100), cv=CV),
    "Lasso (CV)":   LassoCV(alphas=np.logspace(-3, 1, 100), max_iter=50000, cv=CV,
                            random_state=0),
    "ElasticNet (CV)": ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.99],
                                    alphas=np.logspace(-3, 1, 60), max_iter=50000,
                                    cv=CV, random_state=0),
}
res = []
for name, est in candidates.items():
    pipe = make_pipeline(StandardScaler(), est).fit(Xa, ya)
    cf = pipe[-1].coef_
    res.append({
        "model": name,
        "alpha": round(float(getattr(pipe[-1], "alpha_", np.nan)), 4),
        "train_RMSE": round(np.sqrt(mean_squared_error(ya, pipe.predict(Xa))), 3),
        "test_RMSE": round(np.sqrt(mean_squared_error(yb, pipe.predict(Xb))), 3),
        "test_R2": round(r2_score(yb, pipe.predict(Xb)), 4),
        "nonzero": int((np.abs(cf) > 1e-8).sum()),
    })
print(pd.DataFrame(res).to_string(index=False))
print(f"\n(n_train = {len(ya)}, p = {p2}, truly informative = {k2})")
print("OLS fits the training data best and generalises worst. Every regularised model wins.")

In [ ]:
# Tuning alpha explicitly, and seeing the whole curve
alphas = np.logspace(-2, 4, 40)
ridge_cv = [cross_val_score(make_pipeline(StandardScaler(), Ridge(alpha=a_)), Xr, yr,
                            cv=CV, scoring="neg_root_mean_squared_error").mean()
            for a_ in alphas]
lasso_cv = [cross_val_score(make_pipeline(StandardScaler(), Lasso(alpha=a_, max_iter=20000)),
                            Xr, yr, cv=CV, scoring="neg_root_mean_squared_error").mean()
            for a_ in alphas]

plt.plot(alphas, -np.array(ridge_cv), "o-", color="steelblue", label="Ridge")
plt.plot(alphas, -np.array(lasso_cv), "o-", color="crimson", label="Lasso")
plt.xscale("log"); plt.xlabel("alpha"); plt.ylabel("cross-validated RMSE")
plt.title("Under-regularised on the left, over-regularised on the right")
plt.legend(); plt.show()

print(f"Best Ridge alpha : {alphas[int(np.argmax(ridge_cv))]:.3f}")
print(f"Best Lasso alpha : {alphas[int(np.argmax(lasso_cv))]:.3f}")
print("As alpha grows without limit, every model converges to 'predict the mean'.")

---
## 1.6 Interpreting a linear model

The great advantage of linear regression is that you can say what it learned. Three rules:

1. **Standardise before comparing magnitudes.** Otherwise you are comparing "per kilogram"
   with "per year".
2. **Coefficients are conditional.** "Holding the other features fixed" — which is a
   fiction if the features are strongly correlated.
3. **Association, not causation** unless the data came from an experiment.

If you modelled $\log y$, the interpretation becomes multiplicative: a coefficient of $0.06$
means about a **6% increase** in $y$ per unit of $x$ (exactly $e^{0.06}-1 = 6.18\%$).

In [ ]:
final = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
coefs = pd.DataFrame({
    "feature": FEATURES,
    "coef_standardised": final[-1].coef_,
    "coef_original_units": LinearRegression().fit(X_train, y_train).coef_,
}).assign(abs_std=lambda d: d.coef_standardised.abs()).sort_values("abs_std", ascending=False)

print(coefs.drop(columns="abs_std").round(1).to_string(index=False))
print()
plt.barh(coefs.feature, coefs.coef_standardised,
         color=["steelblue" if c > 0 else "crimson" for c in coefs.coef_standardised])
plt.axvline(0, color="black", lw=1)
plt.xlabel("coefficient (per standard deviation of the feature)")
plt.title("Feature effects, on a comparable scale")
plt.tight_layout(); plt.show()

print("How to say it out loud:")
print(f"  * one extra square metre is associated with "
      f"{coefs.loc[coefs.feature=='size','coef_original_units'].item():,.0f} more in price,")
print("    holding age and distance fixed")
print(f"  * each additional km from the centre costs about "
      f"{abs(coefs.loc[coefs.feature=='distance','coef_original_units'].item()):,.0f}")
print("  * size has the largest standardised effect, so it is the dominant driver")

In [ ]:
# Log-target interpretation
log_model = make_pipeline(StandardScaler(), LinearRegression()).fit(
    hp[["log_area", "rooms", "luxury"]], np.log(hp.price))
raw_coefs = LinearRegression().fit(hp[["log_area", "rooms", "luxury"]],
                                   np.log(hp.price)).coef_

print("Model: log(price) ~ log(area) + rooms + luxury")
for f_, c in zip(["log_area", "rooms", "luxury"], raw_coefs):
    if f_ == "log_area":
        print(f"  {f_:<9} {c:+.4f}  -> elasticity: a 1% larger area gives {c:.2f}% more price")
    else:
        print(f"  {f_:<9} {c:+.4f}  -> multiplies price by {np.exp(c):.4f} "
              f"({(np.exp(c)-1)*100:+.2f}%)")
print("\nLog-log slopes are ELASTICITIES, which is why economists live on this scale.")

---
## 1.7 Case study: a production-ready pipeline

Real data has mixed types, so we use `ColumnTransformer` to route numeric and categorical
columns down different paths, wrapped in one `Pipeline`. The benefits:

- Preprocessing is fitted **inside** each CV fold, so no leakage (statistics Notebook 10)
- The whole thing is one object: `fit`, `predict`, save, deploy
- Hyperparameters of every step are tunable together with `GridSearchCV`

In [ ]:
# Mixed-type dataset: apartment rentals
m3 = 900
city = rng.choice(["Chennai", "Bengaluru", "Pune", "Kochi"], m3, p=[0.3, 0.35, 0.2, 0.15])
furnish = rng.choice(["none", "semi", "full"], m3, p=[0.4, 0.4, 0.2])
sqft = rng.uniform(400, 2200, m3)
floor = rng.integers(0, 20, m3)
age_yrs = rng.uniform(0, 30, m3)

city_mult = {"Chennai": 1.0, "Bengaluru": 1.25, "Pune": 0.95, "Kochi": 0.8}
furnish_add = {"none": 0, "semi": 3500, "full": 9000}
rent = (np.array([city_mult[c] for c in city]) *
        (6000 + 14*sqft + 300*floor - 120*age_yrs)
        + np.array([furnish_add[f] for f in furnish])
        + rng.normal(0, 3500, m3))

rentals = pd.DataFrame({"city": city, "furnish": furnish, "sqft": sqft,
                        "floor": floor, "age_yrs": age_yrs, "rent": rent})
print(rentals.head())
print(f"\nrent: mean {rentals.rent.mean():,.0f}, sd {rentals.rent.std():,.0f}")

In [ ]:
num_cols = ["sqft", "floor", "age_yrs"]
cat_cols = ["city", "furnish"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
])

pipe = Pipeline([("prep", preprocess), ("model", Ridge(alpha=1.0))])

Xr_, yr_ = rentals.drop(columns="rent"), rentals["rent"]
Xtr, Xte, ytr, yte = train_test_split(Xr_, yr_, test_size=0.2, random_state=0)

pipe.fit(Xtr, ytr)
print(f"Cross-validated R^2 : "
      f"{cross_val_score(pipe, Xtr, ytr, cv=CV, scoring='r2').mean():.4f}")
print(f"Held-out test R^2   : {pipe.score(Xte, yte):.4f}")
print(f"Held-out test RMSE  : {np.sqrt(mean_squared_error(yte, pipe.predict(Xte))):,.0f}")
print(f"Baseline test R^2   : "
      f"{DummyRegressor().fit(Xtr, ytr).score(Xte, yte):.4f}")

In [ ]:
# Tune the whole pipeline at once, including whether to add interactions
grid = GridSearchCV(
    Pipeline([("prep", preprocess),
              ("poly", PolynomialFeatures(degree=1, include_bias=False)),
              ("model", Ridge())]),
    param_grid={"poly__degree": [1, 2],
                "model__alpha": np.logspace(-2, 3, 12)},
    cv=CV, scoring="neg_root_mean_squared_error", n_jobs=1,
).fit(Xtr, ytr)

print(f"Best parameters : {grid.best_params_}")
print(f"Best CV RMSE    : {-grid.best_score_:,.0f}")
print(f"Test RMSE       : {np.sqrt(mean_squared_error(yte, grid.predict(Xte))):,.0f}")
print(f"Test R^2        : {r2_score(yte, grid.predict(Xte)):.4f}")

In [ ]:
# Inspect what the fitted pipeline learned
fitted = pipe.named_steps["prep"]
feat_names = list(fitted.get_feature_names_out())
coef = pipe.named_steps["model"].coef_

impact = (pd.DataFrame({"feature": feat_names, "coefficient": coef})
          .assign(abs_c=lambda d: d.coefficient.abs())
          .sort_values("abs_c", ascending=False))
print(impact.drop(columns="abs_c").round(1).to_string(index=False))

plt.barh(impact.feature[::-1], impact.coefficient[::-1],
         color=["steelblue" if c > 0 else "crimson" for c in impact.coefficient[::-1]])
plt.axvline(0, color="black", lw=1)
plt.xlabel("coefficient"); plt.title("What the rental model learned")
plt.tight_layout(); plt.show()

print("\nBengaluru carries a large positive coefficient and Kochi a negative one, relative")
print("to the dropped reference city -- which matches how the data was generated.")

---
## 1.8 When linear regression is the wrong tool

| Symptom | Better option |
|---|---|
| Residuals show a clear curve you cannot engineer away | trees, gradient boosting, splines |
| Complex interactions between many features | random forest, boosting (Notebook 6) |
| Target is a category | logistic regression (Notebook 2) |
| Target is a count | Poisson regression |
| Target is bounded (0–1, or a proportion) | logistic / beta regression |
| Heavy outliers in $y$ | Huber or quantile regression, or RANSAC |
| Strong multicollinearity and you need coefficients | Ridge, or drop/combine features |
| Data is images, text, audio | representation learning first |

Linear regression is still worth fitting first in every one of these cases, as the baseline.

In [ ]:
# Outliers: least squares vs Huber
xs = np.linspace(0, 10, 80)
ys_clean = 3 + 2*xs + rng.normal(0, 1, 80)
ys = ys_clean.copy()
ys[[5, 20, 40]] += 40                                # three bad measurements

ols_o = LinearRegression().fit(xs.reshape(-1, 1), ys)
hub = HuberRegressor(epsilon=1.35).fit(xs.reshape(-1, 1), ys)

plt.scatter(xs, ys, s=25, color="steelblue", label="data")
plt.scatter(xs[[5, 20, 40]], ys[[5, 20, 40]], s=90, color="crimson", marker="D",
            label="outliers")
plt.plot(xs, ols_o.predict(xs.reshape(-1, 1)), color="crimson", lw=2,
         label=f"OLS (slope {ols_o.coef_[0]:.2f})")
plt.plot(xs, hub.predict(xs.reshape(-1, 1)), color="seagreen", lw=2,
         label=f"Huber (slope {hub.coef_[0]:.2f})")
plt.legend(fontsize=8); plt.title("Squared loss chases outliers; Huber does not")
plt.show()
print(f"True slope: 2.00   OLS: {ols_o.coef_[0]:.3f}   Huber: {hub.coef_[0]:.3f}")

---
## Exercises

**Exercise 1.** Using the bundled diabetes dataset, build and evaluate a linear regression
pipeline. Report CV and test metrics against a baseline, and identify the three most
influential features.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
from sklearn.datasets import load_diabetes

dia = load_diabetes(as_frame=True)
Xd, yd = dia.data, dia.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=0)

pipe_d = make_pipeline(StandardScaler(), LinearRegression()).fit(Xd_tr, yd_tr)
base_d = DummyRegressor().fit(Xd_tr, yd_tr)

print(f"{'model':<28}{'CV R^2':>9}{'test R^2':>10}{'test RMSE':>12}{'test MAE':>10}")
for name, mdl in [("baseline (mean)", base_d), ("linear regression", pipe_d)]:
    cv_ = cross_val_score(mdl, Xd_tr, yd_tr, cv=CV, scoring="r2").mean()
    pr = mdl.predict(Xd_te)
    print(f"{name:<28}{cv_:>9.4f}{r2_score(yd_te, pr):>10.4f}"
          f"{np.sqrt(mean_squared_error(yd_te, pr)):>12.2f}"
          f"{mean_absolute_error(yd_te, pr):>10.2f}")

imp = (pd.DataFrame({"feature": Xd.columns, "coef": pipe_d[-1].coef_})
       .assign(abs_c=lambda d: d.coef.abs()).sort_values("abs_c", ascending=False))
print("\nMost influential features (standardised coefficients):")
print(imp.drop(columns="abs_c").head(5).round(2).to_string(index=False))
print(f"\nThe target ranges {yd.min():.0f}-{yd.max():.0f} with sd {yd.std():.1f}, so an RMSE")
print(f"near 55 means predictions are typically off by about a standard deviation's worth.")
print("This dataset has a genuinely high noise floor -- no linear model will do much better.")

**Exercise 2.** Show that feature engineering beats algorithm choice. Generate data where
`y` depends on `log(x1)`, `x2**2` and `x1*x2`. Compare (a) linear regression on raw features,
(b) a random forest on raw features, (c) linear regression with the right engineered features.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
from sklearn.ensemble import RandomForestRegressor

m4 = 600
x1 = rng.uniform(1, 50, m4)
x2 = rng.uniform(-5, 5, m4)
y4 = 10 + 8*np.log(x1) + 1.5*x2**2 + 0.4*x1*x2 + rng.normal(0, 3, m4)

raw = pd.DataFrame({"x1": x1, "x2": x2})
eng = raw.assign(log_x1=np.log(x1), x2_sq=x2**2, x1_x2=x1*x2)

options = {
    "(a) linear on raw features":      (make_pipeline(StandardScaler(), LinearRegression()), raw),
    "(b) random forest on raw":        (RandomForestRegressor(n_estimators=300, random_state=0), raw),
    "(c) linear on engineered":        (make_pipeline(StandardScaler(), LinearRegression()), eng),
    "(d) random forest on engineered": (RandomForestRegressor(n_estimators=300, random_state=0), eng),
}
print(f"{'approach':<34}{'CV R^2':>10}{'CV RMSE':>10}")
for name, (est, Xuse) in options.items():
    r2_ = cross_val_score(est, Xuse, y4, cv=CV, scoring="r2").mean()
    rmse_ = -cross_val_score(est, Xuse, y4, cv=CV,
                             scoring="neg_root_mean_squared_error").mean()
    print(f"{name:<34}{r2_:>10.4f}{rmse_:>10.3f}")

print(f"\nTrue noise sd = 3.0, so CV RMSE near 3 is the ceiling.")
print("Linear regression with the right three features beats a 300-tree forest on the raw")
print("ones -- and it is faster, smaller and explainable. Feature engineering is leverage.")
print("The forest gains nothing from the engineered columns: it was already approximating")
print("them internally, just less efficiently.")

**Exercise 3.** A colleague reports test $R^2 = 0.94$ for a house-price model that includes
`price_per_sqft` as a feature. Explain the problem, quantify it, and give the corrected
result.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
m5 = 800
sqft5 = rng.uniform(500, 3000, m5)
loc_score = rng.uniform(1, 10, m5)
price5 = 500*sqft5 + 40_000*loc_score + rng.normal(0, 90_000, m5)

leaky = pd.DataFrame({"sqft": sqft5, "loc_score": loc_score,
                      "price_per_sqft": price5 / sqft5})     # LEAK: derived from the target
clean = leaky.drop(columns="price_per_sqft")

for name, Xuse in [("with price_per_sqft (leaky)", leaky), ("without it (correct)", clean)]:
    Xa5, Xb5, ya5, yb5 = train_test_split(Xuse, price5, test_size=0.25, random_state=0)
    mdl = make_pipeline(StandardScaler(), LinearRegression()).fit(Xa5, ya5)
    print(f"  {name:<30} test R^2 = {mdl.score(Xb5, yb5):.4f}, "
          f"RMSE = {np.sqrt(mean_squared_error(yb5, mdl.predict(Xb5))):,.0f}")

print()
print("The problem: price_per_sqft = price / sqft, so the feature CONTAINS the target.")
print("Multiply it by sqft and you recover the answer exactly. The model is not predicting;")
print("it is performing arithmetic on information it should not have.")
print(f"\nThe inflation is about {0.94 - 0.55:.2f} of R^2 -- the difference between a model")
print("that looks deployable and one that honestly reflects what you know beforehand.")
print("\nAt prediction time for an unsold house you do not know its price, so you cannot")
print("compute price_per_sqft. That is the test every feature must pass.")

**Exercise 4 (challenge).** Build the best linear model you can for the rental dataset from
section 1.7 by adding engineered features, and justify each addition with cross-validation.
Then check whether a non-linear model beats it — and decide what you would ship.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
from sklearn.ensemble import GradientBoostingRegressor

def cv_rmse(est, Xuse, yuse=yr_):
    return -cross_val_score(est, Xuse, yuse, cv=CV,
                            scoring="neg_root_mean_squared_error").mean()

step0 = Xr_.copy()
step1 = step0.assign(sqft_per_floor=step0.sqft / (step0.floor + 1))
step2 = step1.assign(log_sqft=np.log(step1.sqft))
step3 = step2.assign(is_new=(step2.age_yrs < 5).astype(int),
                     sqft_x_new=step2.sqft * (step2.age_yrs < 5))

def build(cols_frame):
    numeric = [c for c in cols_frame.columns if cols_frame[c].dtype != object]
    categorical = [c for c in cols_frame.columns if cols_frame[c].dtype == object]
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", StandardScaler(), numeric),
            ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical)])),
        ("model", RidgeCV(alphas=np.logspace(-2, 3, 40), cv=CV)),
    ])

print("Incremental feature engineering, judged by CV RMSE:")
prev = None
for name, frame in [("baseline features", step0),
                    ("+ sqft per floor", step1),
                    ("+ log(sqft)", step2),
                    ("+ is_new and its interaction", step3)]:
    score = cv_rmse(build(frame), frame)
    delta = "" if prev is None else f"   change {score - prev:+.1f}"
    print(f"  {name:<32} CV RMSE = {score:>8,.0f}{delta}")
    prev = score

In [ ]:
print("\nNow compare against non-linear models on the best feature set:")
contenders = {
    "Ridge (engineered features)": build(step3),
    "Random forest":               Pipeline([
        ("prep", ColumnTransformer([
            ("num", "passthrough", ["sqft", "floor", "age_yrs", "sqft_per_floor",
                                    "log_sqft", "is_new", "sqft_x_new"]),
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["city", "furnish"])])),
        ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=0))]),
    "Gradient boosting":           Pipeline([
        ("prep", ColumnTransformer([
            ("num", "passthrough", ["sqft", "floor", "age_yrs", "sqft_per_floor",
                                    "log_sqft", "is_new", "sqft_x_new"]),
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["city", "furnish"])])),
        ("model", GradientBoostingRegressor(random_state=0))]),
}
for name, est in contenders.items():
    print(f"  {name:<32} CV RMSE = {cv_rmse(est, step3):>8,.0f}")

print()
print("What I would ship, and why:")
print("  The rent in this dataset really is close to linear in sqft, floor and age with")
print("  a multiplicative city effect, so the linear model is at or near the noise floor")
print("  (the simulation used sd = 3,500). The tree models cannot beat a correct linear")
print("  specification, and they cost interpretability, model size and inference time.")
print("  Ship the Ridge pipeline. Revisit if residual plots later reveal structure it")
print("  cannot capture -- and keep the boosting model as the benchmark that tells you")
print("  whether any structure is left on the table.")

---
## Summary

| Concept | Key point |
|---|---|
| Model | $\hat{y} = \mathbf{x}^\top\boldsymbol\beta$; minimise MSE |
| Fitting | Normal equation (exact) or gradient descent (scalable) |
| Scaling | Not needed for OLS; **essential** for SGD, Ridge, Lasso, and comparing coefficients |
| Metrics | Report RMSE or MAE **plus** $R^2$, always against a `DummyRegressor` |
| Feature engineering | Logs, powers, interactions, ratios — usually more valuable than changing algorithm |
| Log target | Turns multiplicative effects additive; coefficients become percentages |
| Ridge | Correlated features, keep them all |
| Lasso | Many useless features, want a short list |
| Elastic Net | Both |
| `Pipeline` + `ColumnTransformer` | Mixed types, no leakage, one deployable object |
| Interpretation | Standardise, say "holding others fixed", say "associated with" |
| Wrong tool when | Curved residuals, categorical target, heavy outliers, unstructured data |

**Next up:** [Notebook 2 — Logistic Regression](2.%20Logistic%20Regression.ipynb), the same
machinery adapted to predicting categories.